In [2]:
%uv pip install -U torch==2.12.1 unsloth==2026.9.11 transformers==5.5.0 datasets==4.3.0 accelerate bitsandbytes

Using Python 3.12.6 environment at: /usr/local
Resolved 101 packages in 184ms
Prepared 8 packages in 2ms
Uninstalled 8 packages in 341ms
Installed 8 packages in 411ms
 - cuda-bindings==12.9.4
 + cuda-bindings==13.4.3
 - huggingface-hub==0.36.2
 + huggingface-hub==1.33.0
 - numpy==2.2.6
 + numpy==2.5.3
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.10.0
 + torch==2.12.1
 - torchvision==0.25.0
 + torchvision==0.27.1
 - transformers==4.57.6
 + transformers==5.5.0
 - triton==3.6.0
 + triton==3.7.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
%uv pip install vllm==0.18.0

Using Python 3.12.6 environment at: /usr/local
Resolved 179 packages in 135ms
Uninstalled 8 packages in 350ms
Installed 8 packages in 394ms
 - cuda-bindings==13.4.3
 + cuda-bindings==12.9.4
 - huggingface-hub==1.33.0
 + huggingface-hub==0.36.2
 - numpy==2.5.3
 + numpy==2.2.6
 - setuptools==81.0.0
 + setuptools==80.10.2
 - torch==2.12.1
 + torch==2.10.0
 - torchvision==0.27.1
 + torchvision==0.25.0
 - transformers==5.5.0
 + transformers==4.57.6
 - triton==3.7.1
 + triton==3.6.0
Note: you may need to restart the kernel to use updated packages.


In [42]:
!apt-get update -qq
!apt-get install -y libcurand-dev-12-8




The following additional packages will be installed:
  libcurand-12-8
The following NEW packages will be installed:
  libcurand-12-8 libcurand-dev-12-8
0 upgraded, 2 newly installed, 0 to remove and 151 not upgraded.
Need to get 89.6 MB of archives.
After this operation, 276 MB of additional disk space will be used.
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  libcurand-12-8 10.3.9.90-1 [44.8 MB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64  libcurand-dev-12-8 10.3.9.90-1 [44.8 MB]
Fetched 89.6 MB in 3s (28.1 MB/s)
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package libcurand-12-8.
(Reading database ... 33321 files and directories currently installed.)
Preparing to unpack .../libcurand-12-8_10.3.9.90-1_amd64.deb ...
Unpacking libcurand-12-8 (10.3.9.90-1) ...
Selecting previously unselected package libcurand-dev-12-8.
Preparing to unpack .../libcurand-dev

In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 09-29 10:32:00 [vllm_utils.py:827] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    NVIDIA L40S. Num GPUs = 1. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `u

<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 09-29 10:32:09 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 09-29 10:32:09 [model.py:1582] Using max model len 2048
INFO 09-29 10:32:09 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 09-29 10:32:09 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 09-29 10:32:10 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='unsloth/Qwen3-4B-Base', speculative_config=None, tokenizer='unsloth/Qwen3-4B-Base', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 09-29 10:32:14 [default_loader.py:384] Loading weights took 2.30 seconds
INFO 09-29 10:32:14 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 09-29 10:32:15 [gpu_model_runner.py:4566] Model loading took 7.65 GiB memory and 3.670180 seconds
INFO 09-29 10:32:22 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/bdfa9670f6/rank_0_0/backbone for vLLM's torch.compile
INFO 09-29 10:32:22 [backends.py:1048] Dynamo bytecode transform time: 6.00 s
INFO 09-29 10:32:29 [backends.py:284] Directly load the compiled graph(s) for compile range (1, 6144) from the cache, took 5.554 s
INFO 09-29 10:32:29 [monitor.py:48] torch.compile took 12.99 s in total
INFO 09-29 10:32:29 [decorators.py:296] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/740979c12e47a6704e1777669b44da106c1ef943dbc8027c8e783f4824cbb439/rank_0_0/model
INFO 09-29 10:32:29 [monitor.py:76] Initial profiling/warmup run took 0.16 s
INFO 09-29 10:32:29 [top

...

Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'norm', 'post_layernorm', 'norm2', 'q_norm', 'layer_norm2', 'post_per_layer_input_norm', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/Qwen3-4B-Base and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'post_attention_layernorm', 'cross_attn_post_attention_layernorm', 'input_layernorm', 'ffn_norm', 'norm', 'post_layernorm', 'norm2', 'q_norm', 'cross_attn_input_layernorm', 'layer_norm2', 'post_per_layer_input_norm', 'layer_norm1', 'post_feedforward_layernorm', 'norm1']


Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [2]:
reasoning_start = "<think>"
reasoning_end   = "</think>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <think> and </think>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

In [3]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with out specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

In [4]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <think> and </think>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>What is 1+1?<think>I think it's 2.</think><SOLUTION>2</SOLUTION><|endoftext|>What is 2+2?<think>"

In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


In [6]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated <think> and </think>
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

In [7]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <think> and </think>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<think>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165 = 49 + 14√(x² - 52) + (√(x² -

In [8]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

(63, 5)

In [9]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 63
})

In [10]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 4, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/63 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 63 | Num Epochs = 4 | Total steps = 32
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.12/site-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Step,Training Loss
5,0.780400
10,0.552700
15,0.451000
20,0.387600
25,0.323800
30,0.318100


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


TrainOutput(global_step=32, training_loss=0.4577145166695118, metrics={'train_runtime': 48.3633, 'train_samples_per_second': 5.211, 'train_steps_per_second': 0.662, 'total_flos': 5050271244693504.0, 'train_loss': 0.4577145166695118, 'epoch': 4.0})

In [12]:
dataset[0]["expected_answer"]

'17'

In [13]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

You are given a problem.
Think about the problem and provide your working out.
Place it between <think> and </think>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?<think>Okay, let's see. So the problem is about Jenifer who has 82 cents in pennies and nickels. Her younger brother thought all her nickels were dimes and counted the total as $1.47. We need to find out how many pennies Jenifer has. Hmm, let me break this down.

First, I need to set up some equations. Let's denote the number of pennies as P and the number of nickels as N. Since pennies are worth 1 cent each and nickels are 5 cents each, the total amount Jenifer has is 1P + 5N = 82 cents. That's the first equation.

Now, her brother mistook all the nickels for dimes. Dimes are 10 cents each. So if he counted all the nickels as dimes, the 

In [14]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

0

In [16]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

In [15]:
dataset[0]

NameError: name 'dataset' is not defined

In [17]:
dataset[0]["solution"]

'34'

In [18]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

'34'

In [19]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

{'prompt': [{'role': 'system',
   'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <think> and </think>.\nThen, provide your solution between <SOLUTION></SOLUTION>'},
  {'role': 'user',
   'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ

In [20]:
import re

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

In [21]:
match_format.findall(
    "Let me think!</think>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

['\n2\n']

In [22]:
match_format.findall(
    "<think>Let me think!</think>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

['  2  ']

In [23]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

In [24]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward <start_working_out> since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

In [25]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

In [26]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']


In [27]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

In [28]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <think> and </think>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<think>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  193


In [29]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

In [30]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,728 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


WARNING 09-29 10:36:40 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!
********************Question:
In the diagram, each of the three identical circles touch the other two.  The circumference of each circle is 36.  What is the perimeter of the shaded region? [asy]

defaultpen(1);

path p = (1, 0){down}..{-dir(30)}dir(-60){dir(30)}..{dir(-30)}((2, 0) + dir(-120)){-dir(-30)}..{up}(1, 0)--cycle;
fill(p, gray(0.75));

draw(unitcircle);
draw(shift(2 * dir(-60)) * unitcircle);
draw(shift(2) * unitcircle);
[/asy] 
Answer:
18 
Response:
Okay, let's see. The problem says there are three identical circles touching each other, and each has a circumference of 36. I need to find the perimeter of the shaded region. Hmm, 

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std,rewards / check_numbers / mean,rewards / check_numbers / std
1,0.074700,-3.156250,3.139597,1572.250000,796.000000,1854.000000,0.500000,1290.500000,796.000000,1620.000000,0.143485,0.000000,0.000000,-1.218750,1.471606,-2.000000,0.000000,0.062500,2.780138
2,0.052200,-4.687500,2.424038,1671.937500,957.000000,1854.000000,0.687500,1271.400024,957.000000,1758.000000,0.119262,0.000000,0.000000,-2.062500,1.436141,-2.000000,0.000000,-0.625000,2.872281
3,0.001500,-7.250000,0.500000,1852.187500,1825.000000,1854.000000,0.937500,1825.000000,1825.000000,1825.000000,0.101134,0.000000,0.000000,-2.812500,0.750000,-2.000000,0.000000,-2.437500,0.250000
4,0.000100,-7.500000,0.000000,1854.000000,1854.000000,1854.000000,1.000000,0.000000,0.000000,0.000000,0.097989,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
5,0.011700,-6.562500,1.125000,1821.187500,1585.000000,1854.000000,0.875000,1591.500000,1585.000000,1598.000000,0.123251,0.000000,0.000000,-2.812500,0.750000,-2.000000,0.000000,-1.750000,2.049390
6,0.041500,-6.125000,1.799038,1766.812500,1109.000000,1854.000000,0.812500,1389.000000,1109.000000,1706.000000,0.114895,0.000000,0.000000,-2.437500,1.209339,-2.000000,0.000000,-1.687500,2.040221
7,0.000100,-7.500000,0.000000,1854.000000,1854.000000,1854.000000,1.000000,0.000000,0.000000,0.000000,0.102027,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
8,0.000100,-7.500000,0.000000,1854.000000,1854.000000,1854.000000,1.000000,0.000000,0.000000,0.000000,0.091604,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
9,0.000100,-7.500000,0.000000,1854.000000,1854.000000,1854.000000,1.000000,0.000000,0.000000,0.000000,0.118642,0.000000,0.000000,-3.000000,0.000000,-2.000000,0.000000,-2.500000,0.000000
10,0.027600,-3.843750,1.648147,1707.937500,1223.000000,1854.000000,0.437500,1594.333374,1223.000000,1782.000000,0.129500,0.000000,0.000000,-1.218750,1.471606,-2.000000,0.000000,-0.625000,2.500000


********************Question:
The graph of the equation $x^{y}=y^{x}$ in the first quadrant (i.e., the region where $x>0$ and $y>0$) consists of a straight line and a curve. Find the sum of the coordinates of the intersection points of a straight line and a curve, rounded down. 
Answer:
5 
Response:
Okay, let's see here. The problem is about the graph of the equation \( x^y = y^x \) in the first quadrant, where both \( x \) and \( y \) are positive. We need to find the sum of the coordinates of the intersection points of a straight line and a curve, rounded down. Hmm, let's break this down.

First, I recall that when dealing with equations like \( x^y = y^x \), sometimes it's helpful to take the natural logarithm of both sides to simplify. Let me try that. Taking ln of both sides gives \( y \ln x = x \ln y \). Wait, that's interesting. Let me rearrange that equation.

Dividing both sides by \( xy \), we get \( \frac{\ln x}{x} = \frac{\ln y}{y} \). So the equation becomes \( \frac{\ln x

KeyboardInterrupt: 

In [31]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/17 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, so I need to find the number of subsets of the set {1, 2, 3, ..., 12} that contain exactly one or two prime numbers. Hmm, let's break this down.\n\nFirst, I should identify all the prime numbers in the set {1, 2, ..., 12}. Let me list them out. The primes are 2, 3, 5, 7, and 11. So there are 5 prime numbers. The other numbers are non-prime: 1, 4, 6, 8, 9, 10, 12. That's 7 non-prime numbers.\n\nThe problem is asking for subsets that have either exactly one prime or exactly two primes. So maybe I should calculate each case separately and then add them together.\n\nLet's start with the subsets containing exactly one prime. If I pick one prime, then the other elements in the subset can be any combination of the non-prime numbers. The number of ways to choose one prime is C(5,1) = 5. For each of these choices, the non-prime elements can be any subset the 7 non-primes. The number of subsets of the non-primes is 2^7, since each non-prime can either be included or not. So the total numb

In [32]:
model.save_lora("grpo_saved_lora")

In [33]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [36]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 144?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 4096,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, so I need to find the square root of 144. Hmm, let me think. I remember that the square root of a number is a value that, when multiplied by itself, gives the original number. For example, the square root of 9 is 3 because 3 x 3 is 9. So in this case, I need to find a number that, when multiplied by itself, equals 144.\n\nRight, let me start with the units place. 4 x 4 is 16, which ends with a 6. But 144 ends with a 4. Interesting. Wait, maybe I should consider both 12 and -12 as possible answers. Because both 12 squared and negative 12 squared would give 144. But if the problem doesn't specify, usually when we talk about square roots, it's the principal (positive) root. So that should be 12.\n\nBut let me verify. If I do 12 x 12, that’s 144. Yes, that’s correct. Alternatively, 11 squared is 121… not enough. 13 squared would be 169… too high. So 12 must be the right number. Therefore, the square root of 144 is 12. I think that's it.\nTo find the square root of 144, we need to de

In [39]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "A shop gives a 20% discount on an item priced at ₹500. What is the final price?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'Okay, let\'s break down this problem. We have a shop that offers a 20% discount on an item that\'s originally priced at ₹500. The question is asking for the final price after the discount has been applied. Hmm, right. So basically, I need to find 20% of ₹500 and then subtract that from the original price. Let me think step by step.\n\nFirst, calculating 20% of ₹500. To do that, I remember that percentages are over 100, so 20% would be 20/100 which is the same as 0.20 in decimal form. Then multiply the original price by 0.20. So 500 multiplied by 0.20. Let me compute that. 500 times 0.20 is 100. Right? Yeah, because half of 500 is 250, and 20% is slightly less, so 100. So the discount amount is ₹100. Then, subtracting that from the original price: 500 minus 100 gives ₹400. Therefore, the final price after the discount is ₹400. \n\nWait, but let me just check if there\'s another way or maybe I missed something. Like, sometimes people confuse discount calculations. But yeah, discount is 

In [40]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "A shop gives a 20% discount on an item priced at ₹500. What is the final price?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, let's see here. The problem says there's a shop offering a 20% discount on an item that's priced at ₹500. I need to find out the final price after the discount. Hmm, discounts are usually calculated based on the original price. So, maybe I should find 20% of 500 first and then subtract that from the original price. Let me work that out.\n\nFirst, calculate 20% of ₹500. Since percentage is out of 100, 20% means 20 per hundred. So, dividing 500 by 100 gives 5… correct? Right, 500 divided by 100 equals 5. Then multiply by 20… so 5 multiplied by 20 is 100. So the discount is ₹100. Now, subtract that discount from the original price. That would be 500 minus 100, which is ₹400. So, the final price after the discount is ₹400. \n\nWait, is there another way to do this? Maybe using decimals? Let me think. 20% is the same as 0.20 in decimal. So if I multiply 0.20 by 500, that should give the discount amount, which is ₹100 again. Yeah, 0.20 * 500 is 100, same as before. Then subtracting fr

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "A shop gives a 20% discount on an item priced at ₹500. What is the final price?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output

In [41]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "If x² = 49, what are the possible values of x?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, so the problem is x squared equals 49. Hmm, let me think. I need to find all possible values of x that satisfy this equation. Right, squaring a number means multiplying it by itself. So if x times x equals 49, then x could be either a positive or negative number because both a positive and a negative number, when squared, give a positive result. Let me see, 7 times 7 is 49, so one solution is x = 7. But wait, what about -7? If you square -7, that's (-7) times (-7), which is 49 as well. So both 7 and -7 should be solutions. Yeah, that makes sense. Let me double-check. If I plug 7 into the equation: 7 squared is 49, that works. And negative 7: (-7) squared is 343... no, hold on. Wait, no, (-7) * (-7) is 49. Yeah, subtracting a negative is adding, so -7 * -7 is +49. So both 7 and -7 are solutions. I think that's it. So the possible values of x are 7 and -7.\nTo solve the equation \\( x^2 = 49 \\), we need to find all values of \\( x \\) that satisfy this equation.\n\n1. **Understan

In [42]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the smallest positive integer divisible by both 6 and 8?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, so I need to find the smallest positive integer that's divisible by both 6 and 8. Hmm, let me think. When they ask for the smallest number that's divisible by two other numbers, that sounds like the least common multiple (LCM) right? Like, the LCM of 6 and 8. Let me make sure I remember how to find LCM.\n\nWait, there's another method like prime factorization too. Yeah, sometimes when the numbers are bigger, that might be easier. Let's see. Prime factors of 6 are 2 and 3, and for 8 it's 2 times 2 times 2. So to find the LCM, you take the highest power of each prime that appears in either number. So for 2, the highest power is 2^3 since 8 has three 2s. For 3, which only appears in 6, that's 2^3 and 3^1. So multiplying those together: 2^3 is 8, times 3 is 24. So the LCM of 6 and 8 is 24. Let me check: can 24 be divided by 6? Yes, 24 divided by 6 is 4. And 24 divided by 8 is 3. So yes, 24 is divisible by both. Is there a smaller number that both divide? Let me think, the multiples 

In [43]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Which is larger: 3^5 or 5^3?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)

output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text
output

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Okay, so I need to figure out which one is larger between 3 to the 5th power and 5 to the 3rd power. Hmm, let me think. I remember that when comparing exponents, sometimes plugging in the numbers can help. But 3^5 might get a bit big, right? Because 3 multiplied by itself five times? Let me calculate that step by step.\n\nFirst, 3 times 3 is 9. Then 9 times 3 is 27. Then 27 times 3 is 81. And then 81 times 3 would be 243. So 3^5 is 243. \n\nNow, 5^3. That's 5 times 5 is 25. Then 25 times 5 is 125. So 5^3 is 125. \n\nComparing those two, 243 is clearly larger than 125. So 3^5 is bigger. Wait, but is there another way to check this? Maybe without calculating each step? For example, maybe considering the growth rate of exponents? For larger numbers, it might be exponential, so higher bases might become more significant faster. But here, the exponent is smaller. Wait, but even if the other number had a higher exponent, the base difference might still matter. So, maybe not that straightfor